# Evaluation of Trained RL Agents

This notebook evaluates pre-trained expert agents on test data, computes performance metrics, and generates equity curve and drawdown visualizations.

## 1. Configuration (YOUR INITIAL CODE)

In [ ]:

import sys
import yaml
import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from typing import List, Dict, Any

# Add project root to path
sys.path.append('..')

# Import environment and expert classes
from environment.trading_env import TradingEnv
from experts.ppo_expert import PPOAgent
from experts.sac_expert import SACAgent
from experts.ddqn_expert import DDQNAgent
from experts.tqc_expert import TQCAgent
from experts.td3_expert import TD3Agent
from experts.a2c_expert import A2CAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:
ASSET = "EURUSD"
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/test.csv"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

AGENT_CLASSES = {
    "ppo": PPOAgent,
    "sac": SACAgent,
    "ddqn": DDQNAgent,
    "tqc": TQCAgent,
    "td3": TD3Agent,
    "a2c": A2CAgent,
}

experts = []
for name, cls in AGENT_CLASSES.items():
    features = cfg.get(name, {}).get("features", [])
    if not features:
        print(f"Warning: No features found for {name}, skipping.")
        continue
    agent_path = f"../checkpoints/{ASSET}/{name}_expert.pt"
    experts.append({
        "name": name,
        "class": cls,
        "features": features,
        "agent_path": agent_path,
    })

print(f"Loaded {len(experts)} experts for asset {ASSET}.")
print("Test data path:", TEST_DATA_PATH)
print("\nExperts configured:")
for exp in experts:
    print(f"  {exp['name'].upper()}: {len(exp['features'])} features -> {exp['agent_path']}")

## 2. Helper Functions (from training script)

In [ ]:
def detect_asset_type(asset_name: str) -> str:
    asset_upper = asset_name.upper()
    crypto_suffixes = ("USDT", "BUSD", "USDC", "BTC", "ETH", "BNB",
                       "XRP", "ADA", "DOGE", "SOL", "DOT")
    if any(asset_upper.endswith(s) for s in crypto_suffixes):
        return "crypto"
    forex_bases = ("EUR", "GBP", "USD", "JPY", "CHF", "CAD", "AUD", "NZD")
    if (any(asset_upper.startswith(b) for b in forex_bases) and
            any(asset_upper.endswith(b) for b in forex_bases) and
            len(asset_upper) == 6):
        return "forex"
    return "crypto"

def get_env_params(asset_type: str, expert_id: int) -> dict:
    if asset_type == "forex":
        return dict(
            asset_type="forex",
            contract_size=100000.0,
            commission_rate=1.5 / 100000.0,
            spread_rate=0.00005,
            slippage_rate=0.00001,
            leverage=30,
            maintenance_margin_ratio=0.5,
            initial_balance=10000.0,
            window_size=20,
            reward_scale=1.0,
            clip_reward=(-10.0, 10.0),
            max_position_fraction=0.95,
        )
    else:
        return dict(
            asset_type="crypto",
            contract_size=1.0,
            commission_rate=0.0005,
            spread_rate=0.0002,
            slippage_rate=0.0001,
            leverage=5,
            maintenance_margin_ratio=0.5,
            initial_balance=10000.0,
            window_size=20,
            reward_scale=1.0,
            clip_reward=(-10.0, 10.0),
            max_position_fraction=0.95,
        )

EXPERT_IDS = {"ppo": 1, "sac": 2, "ddqn": 3, "tqc": 4, "td3": 5, "a2c": 6}
CONTINUOUS_EXPERTS = {"sac", "tqc", "td3", "a2c"}

## 3. Load Test Data & Environment Factory

In [ ]:
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"\nTest data shape: {test_df.shape}")
print(f"Columns: {test_df.columns.tolist()}")
print(f"Period: {test_df.index[0]} → {test_df.index[-1]}")

asset_type = detect_asset_type(ASSET)
print(f"Asset type: {asset_type}")

window_size = cfg.get("window_size", 20)

def build_env(features: List[str], expert_name: str) -> TradingEnv:
    expert_id = EXPERT_IDS[expert_name]
    continuous = expert_name in CONTINUOUS_EXPERTS
    env_kwargs = get_env_params(asset_type, expert_id)
    env_kwargs.update({
        "df": test_df,
        "feature_cols": features,
        "expert_id": expert_id,
        "continuous_actions": continuous,
        "window_size": window_size,
    })
    return TradingEnv(**env_kwargs)

## 4. Agent Loader (INSTANTIATE + LOAD)

In [ ]:
def create_and_load_agent(expert_name: str, agent_class, agent_path: str, state_dim: int):
    """
    Instantiate the agent with hyperparameters from the config,
    then load the saved state dict.
    """
    exp_cfg = cfg[expert_name]

    # Determine action dimension (discrete vs continuous)
    # A2C is discrete with 3 actions (short, flat, long)
    if expert_name in ("ppo", "ddqn", "a2c"):
        action_dim = 3
    else:
        action_dim = 1

    # Base parameters common to most agents
    base_kwargs = {
        "state_dim": state_dim,
        "hidden_dim": exp_cfg.get("hidden_dim", 256),
        "activation": exp_cfg.get("activation", "relu"),
        "log_dir": None,
        "device": "cpu",
    }

    # ------------------------------------------
    # Instantiate the specific agent
    # ------------------------------------------
    if expert_name == "ppo":
        agent = PPOAgent(
            **base_kwargs,
            action_dim=action_dim,
            num_layers=exp_cfg.get("num_layers", 2),
            lr=exp_cfg.get("learning_rate", 2.5e-4),
            gamma=exp_cfg.get("gamma", 0.99),
            gae_lambda=exp_cfg.get("gae_lambda", 0.95),
            clip_eps=exp_cfg.get("clip_eps", 0.2),
            entropy_coef=exp_cfg.get("entropy_coef", 0.01),
            value_coef=exp_cfg.get("value_coef", 0.5),
            max_grad_norm=exp_cfg.get("max_grad_norm", 0.5),
            update_epochs=exp_cfg.get("update_epochs", 4),
            batch_size=exp_cfg.get("batch_size", 64),
            rollout_steps=exp_cfg.get("rollout_steps", 2048),
        )

    elif expert_name == "sac":
        agent = SACAgent(
            **base_kwargs,
            action_dim=action_dim,
            num_layers=exp_cfg.get("num_layers", 2),
            lr=exp_cfg.get("learning_rate", 3e-4),
            gamma=exp_cfg.get("gamma", 0.99),
            tau=exp_cfg.get("tau", 0.005),
            alpha=exp_cfg.get("alpha", 0.2),
            buffer_size=exp_cfg.get("buffer_size", 100000),
            batch_size=exp_cfg.get("batch_size", 256),
            warmup_steps=exp_cfg.get("warmup_steps", 1000),
            max_grad_norm=exp_cfg.get("max_grad_norm", 1.0),
            log_std_min=exp_cfg.get("log_std_min", -5),
            log_std_max=exp_cfg.get("log_std_max", 2),
        )

    elif expert_name == "ddqn":
        agent = DDQNAgent(
            **base_kwargs,
            action_dim=action_dim,
            num_layers=exp_cfg.get("num_layers", 2),
            lr=exp_cfg.get("learning_rate", 1e-4),
            gamma=exp_cfg.get("gamma", 0.99),
            batch_size=exp_cfg.get("batch_size", 256),
            buffer_size=exp_cfg.get("buffer_size", 100000),
            target_update_freq=exp_cfg.get("target_update_freq", 1000),
            epsilon_start=exp_cfg.get("epsilon_start", 1.0),
            epsilon_end=exp_cfg.get("epsilon_end", 0.05),
            epsilon_decay=exp_cfg.get("epsilon_decay", 50000),
            dueling=exp_cfg.get("dueling", True),
            warmup_steps=exp_cfg.get("warmup_steps", 1000),
            max_grad_norm=exp_cfg.get("max_grad_norm", 10.0),
        )

    elif expert_name == "tqc":
        agent = TQCAgent(
            **base_kwargs,
            action_dim=action_dim,
            actor_num_layers=exp_cfg.get("actor_num_layers", 2),
            critic_num_layers=exp_cfg.get("critic_num_layers", 2),
            actor_hidden_dim=exp_cfg.get("actor_hidden_dim"),
            critic_hidden_dim=exp_cfg.get("critic_hidden_dim"),
            lr=exp_cfg.get("learning_rate", 3e-4),
            lr_actor=exp_cfg.get("lr_actor"),
            lr_critic=exp_cfg.get("lr_critic"),
            gamma=exp_cfg.get("gamma", 0.99),
            tau=exp_cfg.get("tau", 0.005),
            buffer_size=exp_cfg.get("buffer_size", 100000),
            batch_size=exp_cfg.get("batch_size", 256),
            warmup_steps=exp_cfg.get("warmup_steps", 1000),
            log_std_min=exp_cfg.get("log_std_min", -20),
            log_std_max=exp_cfg.get("log_std_max", 2),
            n_critics=exp_cfg.get("n_critics", 2),
            n_quantiles=exp_cfg.get("n_quantiles", 25),
            top_quantiles_to_drop=exp_cfg.get("top_quantiles_to_drop", 2),
            auto_entropy_tuning=exp_cfg.get("auto_entropy_tuning", True),
            alpha=exp_cfg.get("alpha", 0.2),
            grad_norm_clip=exp_cfg.get("grad_norm_clip", 1.0),
        )

    elif expert_name == "td3":
        agent = TD3Agent(
            **base_kwargs,
            action_dim=action_dim,
            actor_num_layers=exp_cfg.get("actor_num_layers", 2),
            critic_num_layers=exp_cfg.get("critic_num_layers", 2),
            actor_hidden_dim=exp_cfg.get("actor_hidden_dim"),
            critic_hidden_dim=exp_cfg.get("critic_hidden_dim"),
            lr_actor=exp_cfg.get("lr_actor", 3e-4),
            lr_critic=exp_cfg.get("lr_critic", 3e-4),
            gamma=exp_cfg.get("gamma", 0.99),
            tau=exp_cfg.get("tau", 0.005),
            buffer_size=exp_cfg.get("buffer_size", 100000),
            batch_size=exp_cfg.get("batch_size", 256),
            warmup_steps=exp_cfg.get("warmup_steps", 1000),
            exploration_noise=exp_cfg.get("exploration_noise", 0.1),
            policy_noise=exp_cfg.get("policy_noise", 0.2),
            noise_clip=exp_cfg.get("noise_clip", 0.5),
            policy_delay=exp_cfg.get("policy_delay", 2),
            grad_norm_clip=exp_cfg.get("grad_norm_clip", 1.0),
        )

    elif expert_name == "a2c":
        # A2C uses a shared network (ActorCriticNet) with discrete actions (num_actions=3)
        agent = A2CAgent(
            state_dim=state_dim,
            num_actions=3,  # discrete: short, flat, long
            lr=exp_cfg.get("lr", 7e-4),
            gamma=exp_cfg.get("gamma", 0.99),
            gae_lambda=exp_cfg.get("gae_lambda", 0.95),
            entropy_coef=exp_cfg.get("entropy_coef", 0.05),
            entropy_coef_final=exp_cfg.get("entropy_coef_final", 0.001),
            entropy_anneal_steps=exp_cfg.get("entropy_anneal_steps", 500_000),
            value_coef=exp_cfg.get("value_coef", 0.5),
            max_grad_norm=exp_cfg.get("max_grad_norm", 0.5),
            hidden_dim=exp_cfg.get("hidden_dim", 256),
            num_layers=exp_cfg.get("num_layers", 2),
            activation=exp_cfg.get("activation", "tanh"),
            n_steps=exp_cfg.get("n_steps", 128),
            normalize_rewards=exp_cfg.get("normalize_rewards", False),
            normalize_advantages=exp_cfg.get("normalize_advantages", True),
            lr_anneal_steps=exp_cfg.get("lr_anneal_steps", 0),
            log_dir=None,
            device="cpu",
        )

    else:
        raise ValueError(f"Unknown expert: {expert_name}")

    # Load the saved weights (uses the agent's own load method)
    agent.load(agent_path)
    return agent

## 5. Evaluation Function

In [ ]:
def evaluate_agent(agent, env):
    obs, _ = env.reset()
    done = False
    cum_return = [0.0]
    while not done:
        if hasattr(agent, 'predict'):
            action, _ = agent.predict(obs, deterministic=True)
        elif hasattr(agent, 'act'):
            action = agent.act(obs)
        else:
            raise AttributeError(f"Agent {type(agent)} has no 'predict' or 'act'")
        
        # --- FIX: handle tuple returns (e.g., PPO) and ensure integer action ---
        if isinstance(action, tuple):
            action = action[0]                      # extract first element
        if isinstance(action, (np.ndarray, list)):
            action = int(action[0]) if len(action) > 0 else 0
        else:
            action = int(action)                   # cast to int for discrete env
        
        obs, reward, terminated, truncated, info = env.step(action)
        step_return = info.get('step_return', 0.0)
        cum_return.append(cum_return[-1] + step_return)
        done = terminated or truncated
    return np.array(cum_return)

## 6. Evaluate All Experts

In [ ]:
results = {}   # name -> cumulative returns

for exp in tqdm(experts, desc="Evaluating experts"):
    name = exp["name"]
    agent_class = exp["class"]
    features = exp["features"]
    agent_path = exp["agent_path"]

    print(f"\nEvaluating {name.upper()}...")

    # Build environment
    env = build_env(features, name)

    # State dimension: len(features) + 6 (state features from env)
    state_dim = len(features) + 6

    # Create and load the agent
    agent = create_and_load_agent(name, agent_class, agent_path, state_dim)

    # Evaluate
    cum_ret = evaluate_agent(agent, env)
    results[name] = cum_ret
    print(f"  Final cumulative return: {cum_ret[-1]:.4f} ({cum_ret[-1]*100:.2f}%)")

## 7. Performance Metrics

In [ ]:
def compute_metrics(cum_returns):
    """
    Compute performance metrics from cumulative returns (fractional).
    cum_returns[0] = 0 (starting at 0% return).
    """
    daily_returns = np.diff(cum_returns)
    total_return = cum_returns[-1]
    n_days = len(daily_returns)
    
    # Annualized metrics
    ann_return = (1 + total_return) ** (252 / n_days) - 1 if n_days > 0 else 0.0
    ann_vol = np.std(daily_returns) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else 0.0
    
    # --- Correct drawdown calculation ---
    # Convert cumulative returns to equity (starting at 1.0)
    equity = 1 + cum_returns
    # Running maximum of equity
    peak_equity = np.maximum.accumulate(equity)
    # Drawdown = (peak - current) / peak
    drawdown = (peak_equity - equity) / peak_equity
    max_dd = np.max(drawdown)
    # -----------------------------------
    
    win_rate = np.mean(daily_returns > 0)
    
    return {
        'Total Return': total_return,
        'Annualized Return': ann_return,
        'Annualized Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown': max_dd,
        'Win Rate': win_rate,
        'Daily Returns': daily_returns,
        'Drawdown Series': drawdown,
    }

# Compute for all agents
metrics_dict = {}
for name, cum_ret in results.items():
    metrics_dict[name] = compute_metrics(cum_ret)

# Create formatted DataFrame for display
display_df = pd.DataFrame(metrics_dict).T
raw_metrics_df = display_df.copy()   # raw values for saving

pct_cols = ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Max Drawdown', 'Win Rate']
display_df[pct_cols] = display_df[pct_cols].applymap(lambda x: f"{x:.2%}")
display_df['Sharpe Ratio'] = display_df['Sharpe Ratio'].apply(lambda x: f"{x:.2f}")

print("\n=== Performance Metrics ===")
display_df

## 8. Plot Equity Curves

In [ ]:
plt.figure(figsize=(14, 8))
for name, cum_ret in results.items():
    plt.plot(cum_ret, label=name.upper(), linewidth=2)
plt.title(f"Equity Curves on Test Data ({ASSET})", fontsize=16)
plt.xlabel("Time Steps")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Plot Drawdowns

In [ ]:
# Overlaid
plt.figure(figsize=(14, 8))
for name in results.keys():
    drawdown = metrics_dict[name]['Drawdown Series']
    plt.plot(drawdown, label=name.upper(), linewidth=2)
plt.title(f"Drawdown Comparison ({ASSET})", fontsize=16)
plt.xlabel("Time Steps")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Separate subplots
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()
for i, name in enumerate(results.keys()):
    drawdown = metrics_dict[name]['Drawdown Series']
    ax = axes[i]
    ax.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
    ax.plot(drawdown, color='darkred', linewidth=1)
    ax.set_title(f"{name.upper()} Drawdown", fontsize=12)
    ax.set_xlabel("Time Steps")
    ax.set_ylabel("Drawdown")
    ax.grid(True, alpha=0.3)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

## 10. Save Results (Optional)

In [ ]:
# raw_metrics_df.to_csv(f"evaluation_metrics_{ASSET}.csv")

# plt.figure(figsize=(14,8))
# for name, cum_ret in results.items():
#     plt.plot(cum_ret, label=name.upper())
# plt.legend()
# plt.savefig(f"equity_curves_{ASSET}.png", dpi=300)

# plt.figure(figsize=(14,8))
# for name in results.keys():
#     drawdown = metrics_dict[name]['Drawdown Series']
#     plt.plot(drawdown, label=name.upper())
# plt.legend()
# plt.savefig(f"drawdowns_{ASSET}.png", dpi=300)

# print("\nEvaluation complete. Metrics and figures saved.")